# **Studi Komparatif Model Linear Regression dan Random Forest untuk Time Series Forecasting Harga Beras Berdasarkan Klasifikasi Kualitas di Indonesia**

---



#### Install Libraries

In [ ]:
# @title
!pip install pandas
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install scikit-learn
!pip install scipy


#### Import Libraries

In [ ]:
# @title
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
import scipy
import requests

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import norm
from scipy import stats
print("All libraries are ready to go!")

#### **Data Collection**

Title: Monthly Average National Consumer Food Prices (February 2026
Figures)

Url: https://data.badanpangan.go.id/datasetpublications/ary/rata-rata-harga-pangan-bulanan-konsumen-nasional

JSON: https://data.badanpangan.go.id/download/document/dataset/327/1772769168.json/json/json

In [ ]:
import requests
import pandas as pd

# 1. Get the data
url = "https://data.badanpangan.go.id/download/document/dataset/327/1772769168.json/json/json"
try:
    response = requests.get(url, timeout=10) # Added a timeout for better practice
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    data = response.json()

    # Convert the 'data' dictionary into a list of dictionaries where each item is a row.
    df_list = [v for k, v in data.items()]
    df_raw = pd.DataFrame(df_list)

    # Find the row that contains the actual headers (e.g., 'Komoditas', 'Tahun', 'Bulan', 'Harga')
    # This assumes the header row can be identified by the presence of 'Komoditas' in its first column
    header_row_index = None
    for i, row in df_raw.iterrows():
        if row.iloc[0] == 'Komoditas':
            header_row_index = i
            break

    if header_row_index is not None:
        # Re-creating df from the `data` dictionary, using the keys, and then filter by index.
        temp_df = pd.DataFrame.from_dict(data, orient='index')

        # Find the row that contains the actual headers (e.g., 'Komoditas', 'Tahun', 'Bulan', 'Harga')
        header_row_index_in_temp_df = None
        for i, row in temp_df.iterrows():
            if row.iloc[0] == 'Komoditas':
                header_row_index_in_temp_df = i
                break

        if header_row_index_in_temp_df is not None:
            # Set column headers from the identified header row
            df = temp_df.loc[header_row_index_in_temp_df:].copy()
            df.columns = df.iloc[0] # Set the first row (which is the header row) as columns
            df = df[1:].reset_index(drop=True) # Drop the header row itself and reset index

            # Rename columns to remove potential leading/trailing spaces or special characters
            df.columns = df.columns.str.strip().str.replace(' ', '_')

            # FIX: Strip whitespace from 'Komoditas' column values immediately after df creation
            if 'Komoditas' in df.columns:
                df['Komoditas'] = df['Komoditas'].str.strip()

            print("Data downloaded and processed successfully.")
        else:
            print("Error: Could not find header row in the downloaded JSON data.")

    else:
        print("Error: Could not find header row in the downloaded JSON data after initial pass.")

except requests.exceptions.HTTPError as errh:
    print (f"Http Error: {errh}")
except requests.exceptions.ConnectionError as errc:
    print (f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print (f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print (f"Oops: Something Else {err}")
except Exception as e:
    print(f"An unexpected error occurred during data processing: {e}")

#### Display Sample of Data Pangan after Data Downloaded

In [ ]:
# Display All Food Data and Verified the Total count of Record
print(f"Total records in df: {len(df)}")

# Print distinct years and months
if 'Tahun' in df.columns:
    distinct_years = sorted(df['Tahun'].unique().tolist())
    print(f"Distinct Years in df: {distinct_years}")
if 'Bulan' in df.columns:
    distinct_months = sorted(df['Bulan'].unique().tolist())
    print(f"Distinct Months in df: {distinct_months}")

display(df.head())

 #### Filter Data Pangan containing BERAS as Research Scope as Beras Premium,Beras Medium, Beras SPHP

In [ ]:
# @title
# Define the target commodities for filtering
komoditas_targets = ['Beras Premium', 'Beras Medium', 'Beras SPHP']
beras_df = df[df['Komoditas'].isin(komoditas_targets)].copy()
print("Filtered 'beras_df' created for further preprocessing with specific commodities.")


#### Display Beras Sample Data

In [ ]:
# @title
print("Distinct Komoditas in beras_df:", beras_df['Komoditas'].unique())
display(beras_df.head())

### **Data Pre-Processing**
#### **Data Transformation**

#### Converting the 'Harga' (Price) column to a Numeric format.

In [ ]:
beras_df['Harga_Numeric'] = beras_df['Harga'].str.replace('Rp', '', regex=False).str.replace(',', '', regex=False).astype(float)
print("'Harga' column converted to 'Harga_Numeric' (float).")

#### Creating a Date column, Defining a month mapping and setting it as the DataFrame index.

In [ ]:
month_mapping = {
    'Januari': 1, 'Februari': 2, 'Maret': 3, 'April': 4, 'Mei': 5, 'Juni': 6,
    'Juli': 7, 'Agustus': 8, 'September': 9, 'Oktober': 10, 'November': 11, 'Desember': 12
}
print("Month mapping defined.")

# Check if 'Bulan' column exists before trying to access it
if 'Bulan' in beras_df.columns and 'Tahun' in beras_df.columns and 'Harga' in beras_df.columns:
    beras_df['Bulan_Numeric'] = beras_df['Bulan'].str.strip().map(month_mapping)
    beras_df['Date'] = pd.to_datetime(beras_df['Tahun'].astype(str) + '-' + beras_df['Bulan_Numeric'].astype(str) + '-01')
    beras_df = beras_df.set_index('Date').sort_index()
    beras_df = beras_df.drop(columns=['Tahun', 'Bulan', 'Harga', 'Bulan_Numeric'], errors='ignore')
    print("'Date' column created and set as index. Original date/price columns dropped.")
else:
    print("Skipping date creation and column drops as 'Bulan', 'Tahun', or 'Harga' columns are already missing or processed.")

print(beras_df.head())

#### **Data Cleaning**

#### Cek Data Consistency: Cleaning the 'Komoditas' (Commodity) column for

In [ ]:
# @title
beras_df['Komoditas'] = beras_df['Komoditas'].str.strip()
print("'Komoditas' column cleaned for leading/trailing spaces.")

#### Check Missing Values

In [ ]:
# @title
# Check for data Missing Values
beras_df["Harga_Numeric"].isnull().sum()

#### Check Data Duplicates

In [ ]:
# @title
import pandas as pd

# Check for data duplicates
# Create a temporary DataFrame for duplicate checking, including the 'Date' index as a column.
temp_df_for_duplicates = beras_df.reset_index()

# Find duplicates based on 'Komoditas' and 'Date'
duplicates_mask = temp_df_for_duplicates.duplicated(subset=['Komoditas', 'Date'], keep=False)
duplicates = beras_df[duplicates_mask.values] # Apply the mask to the original beras_df

if not duplicates.empty:
    print("Duplicate data found based on Commodity and Date:")
    # Display duplicates, resetting the index to show the 'Date' column for sorting
    display(duplicates.reset_index().sort_values(by=['Komoditas', 'Date']))
    print(f"Number of duplicates: {len(duplicates)}")
    # Optional: If you want to drop duplicates, use (uncomment and adjust if needed):
    # beras_df = beras_df[~duplicates_mask.values].copy() # To remove all duplicate rows
    # print("Duplicates have been removed.")
else:
    print("No duplicate data found based on Commodity and Date.")

#### Check Data with Negative Values

In [ ]:
# @title
# Check for negative values in 'Harga_Numeric'
negative_prices = beras_df[beras_df['Harga_Numeric'] < 0]

if not negative_prices.empty:
    print("Negative values found in 'Harga_Numeric':")
    display(negative_prices)
    print(f"Number of negative values: {len(negative_prices)}")
else:
    print("No negative values found in 'Harga_Numeric'.")

### **Exploratory Data Analysis (EDA)**



#### Find more information about the features and types

In [ ]:
# @title
beras_df.info()

####

#### Find more information about statistical summary of the Harga Beras

In [ ]:
# @title
beras_df["Harga_Numeric"].describe()

#### Create data frame as to show only Beras Premium/Medium/SPHB entries

In [ ]:
# @title
# Filter beras_df to show only 'Beras Premium' entries using the original 'Komoditas' column
beras_premium_df = beras_df[beras_df['Komoditas'] == 'Beras Premium'].copy()

print(f"Total records in beras_premium_df: {len(beras_premium_df)}")

if not beras_premium_df.empty:
    distinct_years = beras_premium_df.index.year.unique().tolist()
    distinct_months = beras_premium_df.index.month.unique().tolist()
    print(f"Distinct Years for Beras Premium: {sorted(distinct_years)}")
    print(f"Distinct Months for Beras Premium: {sorted(distinct_months)}")
else:
    print("No data found for Beras Premium.")

print("First 5 rows of 'Beras Premium' data:")
display(beras_premium_df.head())

In [ ]:
# @title
# Filter beras_df to show only 'Beras Medium' entries using the original 'Komoditas' column
beras_medium_df = beras_df[beras_df['Komoditas'] == 'Beras Medium'].copy()

print(f"Total records in beras_medium_df: {len(beras_medium_df)}")

if not beras_medium_df.empty:
    distinct_years = beras_medium_df.index.year.unique().tolist()
    distinct_months = beras_medium_df.index.month.unique().tolist()
    print(f"Distinct Years for Beras Medium: {sorted(distinct_years)}")
    print(f"Distinct Months for Beras Medium: {sorted(distinct_months)}")
else:
    print("No data found for Beras Medium.")

print("First 5 rows of 'Beras Medium' data:")
display(beras_medium_df.head())

In [ ]:
# @title
# Filter beras_df to show only 'Beras SPHP' entries using the original 'Komoditas' column
beras_sphp_df = beras_df[beras_df['Komoditas'] == 'Beras SPHP'].copy()

print(f"Total records in beras_sphp_df: {len(beras_sphp_df)}")

if not beras_sphp_df.empty:
    distinct_years = beras_sphp_df.index.year.unique().tolist()
    distinct_months = beras_sphp_df.index.month.unique().tolist()
    print(f"Distinct Years for Beras SPHP: {sorted(distinct_years)}")
    print(f"Distinct Months for Beras SPHP: {sorted(distinct_months)}")
else:
    print("No data found for Beras SPHP.")

print("First 5 rows of 'Beras SPHP' data:")
display(beras_sphp_df.head())

#### Discover outliers using Uni-variate analysis

In [ ]:
# This cell specifically Uni-variate analysi focuses on 'All type of Beras'.
if 'Harga_Numeric' not in beras_df.columns:
    # Clean 'Komoditas' column (from cell 65cbeaa4)
    beras_df['Komoditas'] = beras_df['Komoditas'].str.strip()

    # Convert 'Harga' to 'Harga_Numeric' (from cell d1de5fb2)
    beras_df['Harga_Numeric'] = beras_df['Harga'].str.replace('Rp', '', regex=False).str.replace(',', '', regex=False).astype(float)

    # Create 'Date' column and set as index (from cell 5f4bd88c)
    if 'Tahun' in beras_df.columns and 'Bulan' in beras_df.columns:
        beras_df['Bulan_Numeric'] = beras_df['Bulan'].str.strip().map(month_mapping)
        beras_df['Date'] = pd.to_datetime(beras_df['Tahun'].astype(str) + '-' + beras_df['Bulan_Numeric'].astype(str) + '-01')
        beras_df = beras_df.set_index('Date').sort_index()
        beras_df = beras_df.drop(columns=['Tahun', 'Bulan', 'Harga', 'Bulan_Numeric'], errors='ignore')

# Now plot the boxplot using the cleaned numeric column
sns.boxplot(x=beras_df['Harga_Numeric'])
plt.title('Boxplot of Beras Harga (Numeric)')
plt.xlabel('Harga (IDR)')
plt.show()

#### Discover outliers using Uni-variate analysis - Beras Premium

In [ ]:
# This cell specifically Uni-variate analysi focuses on 'Beras Premium'.

# Get the DataFrame for 'Beras Premium' by filtering beras_df using original 'Komoditas' column
beras_premium_df = beras_df[beras_df['Komoditas'] == 'Beras Premium'].copy()

# Check if the DataFrame is not empty and has the 'Harga_Numeric' column
if not beras_premium_df.empty and 'Harga_Numeric' in beras_premium_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=beras_premium_df['Harga_Numeric'])
    plt.title('Boxplot of Beras Premium Harga (Numeric)')
    plt.xlabel('Harga (IDR)')
    plt.show()
else:
    print("Error: 'Beras Premium' data not found or 'Harga_Numeric' column missing after filtering from beras_df.")

Discover outliers using Uni-variate analysis - Beras Medium

In [ ]:
# @title
# This cell focuses on 'Beras Medium' for univariate outlier analysis.

# Get the DataFrame for 'Beras Medium' by filtering beras_df using one-hot encoded column
beras_medium_df = beras_df[beras_df['Komoditas'] == 'Beras Medium']


# Check if the DataFrame is not empty and has the 'Harga_Numeric' column
if not beras_medium_df.empty and 'Harga_Numeric' in beras_medium_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=beras_medium_df['Harga_Numeric'])
    plt.title('Boxplot of Beras Medium Harga (Numeric)')
    plt.xlabel('Harga (IDR)')
    plt.show()
else:
    print("Error: 'Beras Medium' data not found or 'Harga_Numeric' column missing after filtering from beras_df.")

Discover outliers using Uni-variate analysis - Beras SPHP

In [ ]:
# @title
# This cell focuses on 'Beras SPHP' for univariate outlier analysis.

# Get the DataFrame for 'Beras SPHP' by filtering beras_df using one-hot encoded column
beras_sphp_df = beras_df[beras_df['Komoditas'] == 'Beras SPHP']

# Check if the DataFrame is not empty and has the 'Harga_Numeric' column
if not beras_sphp_df.empty and 'Harga_Numeric' in beras_sphp_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=beras_sphp_df['Harga_Numeric'])
    plt.title('Boxplot of Beras SPHP Harga (Numeric)')
    plt.xlabel('Harga (IDR)')
    plt.show()
else:
    print("Error: 'Beras SPHP' data not found or 'Harga_Numeric' column missing after filtering from beras_df.")

#### Discover outliers using Multi-variate analysis - All Type of Beras

In [ ]:
beras_df['Tahun_Numeric'] = beras_df.index.year

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Tahun_Numeric', y='Harga_Numeric', data=beras_df)
plt.title('Scatter Plot of Harga vs. Tahun for Beras (Multi-variate Outlier Analysis)')
plt.xlabel('Tahun')
plt.ylabel('Harga (IDR)')
plt.grid(True)
plt.show()

#### Discover outliers using Multi-variate analysis - Beras Premium

In [ ]:
# @title
# Discover outliers using Multi-variate analysis for Beras Premium

# Create beras_premium_df by filtering beras_df for 'Beras Premium' using one-hot encoded column.
beras_premium_df = beras_df[beras_df['Komoditas'] == 'Beras Premium'].copy()

# Ensure the DataFrame is sorted by Date (index) for correct lag calculation
beras_premium_df = beras_premium_df.sort_index()

# Calculate the lagged price, which is required for the scatter plot
beras_premium_df['Harga_Lag1'] = beras_premium_df['Harga_Numeric'].shift(1)

# Drop any rows where Harga_Lag1 is NaN (i.e., the first month, as no previous value exists)
beras_premium_df = beras_premium_df.dropna(subset=['Harga_Lag1'])

if not beras_premium_df.empty and 'Harga_Numeric' in beras_premium_df.columns and 'Harga_Lag1' in beras_premium_df.columns:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x='Harga_Lag1', y='Harga_Numeric', data=beras_premium_df)
    plt.title('Scatter Plot: Harga_Numeric vs Harga_Lag1 for Beras Premium')
    plt.xlabel('Harga Bulan Sebelumnya (IDR)')
    plt.ylabel('Harga Saat Ini (IDR)')
    plt.grid(True)
    plt.show()
else:
    print("Error: Filtered data for 'Beras Premium' (stored in beras_premium_df) is empty or essential columns are not available after lag calculation.")

#### Discover outliers using Multi-variate analysis - Beras Medium

In [ ]:
# @title
# Discover outliers using Multi-variate analysis for Beras Medium

# Create beras_medium_df by filtering beras_df for 'Beras Medium' using one-hot encoded column.
beras_medium_df = beras_df[beras_df['Komoditas'] == 'Beras Medium'].copy()

# Ensure the DataFrame is sorted by Date (index) for correct lag calculation
beras_medium_df = beras_medium_df.sort_index()

# Calculate the lagged price, which is required for the scatter plot
beras_medium_df['Harga_Lag1'] = beras_medium_df['Harga_Numeric'].shift(1)

# Drop any rows where Harga_Lag1 is NaN (i.e., the first month, as no previous value exists)
beras_medium_df = beras_medium_df.dropna(subset=['Harga_Lag1'])

if not beras_medium_df.empty and 'Harga_Numeric' in beras_medium_df.columns and 'Harga_Lag1' in beras_medium_df.columns:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x='Harga_Lag1', y='Harga_Numeric', data=beras_medium_df)
    plt.title('Scatter Plot: Harga_Numeric vs Harga_Lag1 for Beras Medium')
    plt.xlabel('Harga Bulan Sebelumnya (IDR)')
    plt.ylabel('Harga Saat Ini (IDR)')
    plt.grid(True)
    plt.show()
else:
    print("Error: Filtered data for 'Beras Medium' is empty or essential columns are not available after lag calculation.")

#### Discover outliers using Multi-variate analysis - Beras SPHP

In [ ]:
# @title
# Discover outliers using Multi-variate analysis for Beras SPHP

# Create beras_sphp_df by filtering beras_df for 'Beras SPHP' using one-hot encoded column.
beras_sphp_df = beras_df[beras_df['Komoditas'] == 'Beras SPHP'].copy()

# Ensure the DataFrame is sorted by Date (index) for correct lag calculation
beras_sphp_df = beras_sphp_df.sort_index()

# Calculate the lagged price, which is required for the scatter plot
beras_sphp_df['Harga_Lag1'] = beras_sphp_df['Harga_Numeric'].shift(1)

# Drop any rows where Harga_Lag1 is NaN (i.e., the first month, as no previous value exists)
beras_sphp_df = beras_sphp_df.dropna(subset=['Harga_Lag1'])

if not beras_sphp_df.empty and 'Harga_Numeric' in beras_sphp_df.columns and 'Harga_Lag1' in beras_sphp_df.columns:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x='Harga_Lag1', y='Harga_Numeric', data=beras_sphp_df)
    plt.title('Scatter Plot: Harga_Numeric vs Harga_Lag1 for Beras SPHP')
    plt.xlabel('Harga Bulan Sebelumnya (IDR)')
    plt.ylabel('Harga Saat Ini (IDR)')
    plt.grid(True)
    plt.show()
else:
    print("Error: Filtered data for 'Beras SPHP' is empty or essential columns are not available after lag calculation.")

#### Discover outliers using Histogram analysis - All type of Beras

In [ ]:
# This cell specifically focuses on 'All type of Beras' for histogram analysis.

# Check if the DataFrame is not empty and has the 'Harga_Numeric' column
if not beras_df.empty and 'Harga_Numeric' in beras_df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(beras_df['Harga_Numeric'], kde=True, bins=15)
    plt.title('Histogram of All Beras Types Harga (Numeric)')
    plt.xlabel('Harga (IDR)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
else:
    print("Error: 'beras_df' is empty or 'Harga_Numeric' column is missing.")

#### Discover outliers using Histogram analysis - Beras Premium

In [ ]:
# @title
# This cell specifically focuses on 'Beras Premium' for histogram analysis.

# Create beras_premium_df by filtering beras_df for 'Beras Premium' using one-hot encoded column.
beras_premium_df = beras_df[beras_df['Komoditas'] == 'Beras Premium'].copy()

# Check if the DataFrame is not empty and has the 'Harga_Numeric' column
if not beras_premium_df.empty and 'Harga_Numeric' in beras_premium_df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(beras_premium_df['Harga_Numeric'], kde=True, bins=15)
    plt.title('Histogram of Beras Premium Harga (Numeric)')
    plt.xlabel('Harga (IDR)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
else:
    print("Error: 'Beras Premium' data not found or 'Harga_Numeric' column missing after filtering from beras_df.")

#### Discover outliers using Histogram analysis - Beras Medium

In [ ]:
# @title
# This cell specifically focuses on 'Beras Medium' for histogram analysis.

# Create beras_medium_df by filtering beras_df for 'Beras Medium' using one-hot encoded column.
beras_medium_df = beras_df[beras_df['Komoditas'] == 'Beras Medium'].copy()

# Check if the DataFrame is not empty and has the 'Harga_Numeric' column
if not beras_medium_df.empty and 'Harga_Numeric' in beras_medium_df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(beras_medium_df['Harga_Numeric'], kde=True, bins=15)
    plt.title('Histogram of Beras Medium Harga (Numeric)')
    plt.xlabel('Harga (IDR)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
else:
    print("Error: 'Beras Medium' data not found or 'Harga_Numeric' column missing after filtering from beras_df.")

#### Discover outliers using Histogram  analysis - Beras SPHP

In [ ]:
# @title
# This cell specifically focuses on 'Beras SPHP' for histogram analysis.

# Create beras_sphp_df by filtering beras_df for 'Beras SPHP' using one-hot encoded column.
beras_sphp_df = beras_df[beras_df['Komoditas'] == 'Beras SPHP'].copy()

# Check if the DataFrame is not empty and has the 'Harga_Numeric' column
if not beras_sphp_df.empty and 'Harga_Numeric' in beras_sphp_df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(beras_sphp_df['Harga_Numeric'], kde=True, bins=15)
    plt.title('Histogram of Beras SPHP Harga (Numeric)')
    plt.xlabel('Harga (IDR)')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
else:
    print("Error: 'Beras SPHP' data not found or 'Harga_Numeric' column missing after filtering from beras_df.")

#### Feature Engineering

In [ ]:
# One-hot encode the 'Komoditas' column
beras_df = pd.get_dummies(beras_df, columns=['Komoditas'], prefix='Komoditas', dtype=int)
print("One-hot encoding applied to 'Komoditas' column.")

# Extract time-based features from the 'Date' index
beras_df['Year'] = beras_df.index.year
beras_df['Month'] = beras_df.index.month
beras_df['DayOfWeek'] = beras_df.index.dayofweek # Monday=0, Sunday=6
print("Extracted 'Year', 'Month', and 'DayOfWeek' from the index.")

# Ensure the DataFrame is sorted by Date (index) for correct lag and rolling calculations within groups
beras_df = beras_df.sort_index()

# Calculate lagged price ('Harga_Lag1') for each 'Komoditas' group
# To do this, we need to temporarily recreate the 'Komoditas' column from one-hot encoded columns
commodity_cols = [col for col in beras_df.columns if col.startswith('Komoditas_')]
if commodity_cols:
    # Create a temporary 'Komoditas_Original' column for grouping
    beras_df['Komoditas_Original'] = beras_df[commodity_cols].idxmax(axis=1).str.replace('Komoditas_', '')

    beras_df['Harga_Lag1'] = beras_df.groupby('Komoditas_Original')['Harga_Numeric'].shift(1)
    print("Calculated 'Harga_Lag1' for each commodity.")

    # Calculate 3-period rolling average price ('Harga_RollMean3') for each 'Komoditas' group
    beras_df['Harga_RollMean3'] = beras_df.groupby('Komoditas_Original')['Harga_Numeric'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    print("Calculated 'Harga_RollMean3' for each commodity.")

    # Calculate 3-period rolling standard deviation price ('Harga_RollStd3') for each 'Komoditas' group
    beras_df['Harga_RollStd3'] = beras_df.groupby('Komoditas_Original')['Harga_Numeric'].transform(lambda x: x.rolling(window=3, min_periods=1).std())
    print("Calculated 'Harga_RollStd3' for each commodity.")

    # Drop the temporary 'Komoditas_Original' column
    beras_df = beras_df.drop(columns=['Komoditas_Original'])
else:
    print("Warning: No 'Komoditas_' columns found after one-hot encoding.")

# Drop any rows that contain NaN values introduced by lagged/rolling calculations
# This addresses 'd. Penanganan Nilai Hilang'
initial_rows = len(beras_df)
beras_df = beras_df.dropna()
print(f"Dropped {initial_rows - len(beras_df)} rows with NaN values after feature engineering.")

print("\nDataFrame after feature engineering:")
display(beras_df.head())

### Data Splitting

In [ ]:
# Separate the target variable (y) from the features (X)
y = beras_df['Harga_Numeric'].copy()
X = beras_df.drop('Harga_Numeric', axis=1).copy()

print("Features (X) and target (y) have been separated.")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

# Determine the index for splitting the data (80% for training)
split_index = int(len(X) * 0.8)
print(f"Calculated split index for 80% training: {split_index}")

# Create the training feature set X_train and target set y_train
X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()

# Create the testing feature set X_test and target set y_test
X_test = X.iloc[split_index:].copy()
y_test = y.iloc[split_index:].copy()

# Print the shapes of X_train, X_test, y_train, and y_test
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

### Feature Scaling

Menerapkan penskalaan fitur (misalnya, StandardScaler atau MinMaxScaler) ke fitur numerik dari set pelatihan dan pengujian untuk meningkatkan kinerja model, terutama untuk Linear Regression.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify the numerical features in X_train and X_test that need scaling
# Exclude one-hot encoded columns and keep only the newly created numeric features
numerical_features_to_scale = ['Year', 'Month', 'DayOfWeek', 'Harga_Lag1', 'Harga_RollMean3', 'Harga_RollStd3']

# Ensure these features exist in X_train
existing_numerical_features = [col for col in numerical_features_to_scale if col in X_train.columns]
if len(existing_numerical_features) != len(numerical_features_to_scale):
    print(f"Warning: Some numerical features to scale not found in X_train: {set(numerical_features_to_scale) - set(existing_numerical_features)}")

# Create a StandardScaler object
scaler = StandardScaler()
print("StandardScaler object created.")

# Fit the scaler on the numerical columns of X_train only and transform X_train
X_train[existing_numerical_features] = scaler.fit_transform(X_train[existing_numerical_features])
print("Scaler fitted on X_train numerical features and X_train transformed.")

# Transform the numerical columns of X_test using the fitted scaler
X_test[existing_numerical_features] = scaler.transform(X_test[existing_numerical_features])
print("X_test numerical features transformed using the fitted scaler.")

# Display the first few rows of the scaled X_train and X_test DataFrames
print("\nFirst 5 rows of scaled X_train:")
display(X_train.head())

print("\nFirst 5 rows of scaled X_test:")
display(X_test.head())

### **Model Development & Evaluation **

#### Model Linear Regression

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
import pandas as pd

# --- 3.2.1 Pelatihan Model Linear Regression ---
print("\n--- 3.2.1 Pelatihan Model Linear Regression ---")
# Inisialisasi model Linear Regression
linear_reg_model = LinearRegression()

# Latih model menggunakan data pelatihan yang sudah diskalakan
linear_reg_model.fit(X_train, y_train)

print("Model Linear Regression berhasil dilatih.")

# --- Prediksi pada data uji ---
y_pred_lr = linear_reg_model.predict(X_test)

# --- b. Evaluasi kinerja menggunakan MAE, RMSE, MAPE, dan R² ---
print("\n--- Evaluasi Kinerja Model Linear Regression pada Data Uji ---")
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

# Calculate MAPE manually, handling potential division by zero
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero by replacing 0s in y_true with a small epsilon or filtering
    non_zero_mask = y_true != 0
    return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100

mape_lr = mean_absolute_percentage_error(y_test, y_pred_lr)

print(f"Mean Absolute Error (MAE): {mae_lr:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_lr:.2f}")
print(f"R-squared (R²): {r2_lr:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape_lr:.2f}%")

# --- c. Cross-validation khusus deret waktu untuk memastikan model stabil ---
print("\n--- Time Series Cross-Validation untuk Linear Regression ---")
tscv = TimeSeriesSplit(n_splits=5) # Example: 5 splits

# Combine X and y for cross-validation
X_combined = pd.concat([X_train, X_test])
y_combined = pd.concat([y_train, y_test])

mae_scores_lr = []
rmse_scores_lr = []
r2_scores_lr = []
mape_scores_lr = []

for train_index, val_index in tscv.split(X_combined):
    X_train_cv, X_val_cv = X_combined.iloc[train_index], X_combined.iloc[val_index]
    y_train_cv, y_val_cv = y_combined.iloc[train_index], y_combined.iloc[val_index]

    model_cv = LinearRegression()
    model_cv.fit(X_train_cv, y_train_cv)
    y_pred_cv = model_cv.predict(X_val_cv)

    mae_scores_lr.append(mean_absolute_error(y_val_cv, y_pred_cv))
    rmse_scores_lr.append(np.sqrt(mean_squared_error(y_val_cv, y_pred_cv)))
    r2_scores_lr.append(r2_score(y_val_cv, y_pred_cv))
    mape_scores_lr.append(mean_absolute_percentage_error(y_val_cv, y_pred_cv))

print(f"Average MAE (CV): {np.mean(mae_scores_lr):.2f} (+/- {np.std(mae_scores_lr):.2f})")
print(f"Average RMSE (CV): {np.mean(rmse_scores_lr):.2f} (+/- {np.std(rmse_scores_lr):.2f})")
print(f"Average R-squared (CV): {np.mean(r2_scores_lr):.4f} (+/- {np.std(r2_scores_lr):.4f})")
print(f"Average MAPE (CV): {np.mean(mape_scores_lr):.2f}% (+/- {np.std(mape_scores_lr):.2f}%)")

# --- d. Visualisasi hasil, dengan membandingkan data aktual dan hasil prediksi ---
print("\n--- Visualisasi Prediksi Linear Regression vs. Aktual ---")
# Create a DataFrame for easy plotting with original index
results_lr = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_lr}, index=y_test.index)

plt.figure(figsize=(14, 7))
plt.plot(results_lr.index, results_lr['Actual'], label='Harga Aktual', marker='o', linestyle='-', markersize=4)
plt.plot(results_lr.index, results_lr['Predicted'], label='Prediksi Linear Regression', marker='x', linestyle='--', markersize=4)
plt.title('Perbandingan Harga Aktual dan Prediksi Linear Regression')
plt.xlabel('Tanggal')
plt.ylabel('Harga (IDR)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### Model Random Forest

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
import pandas as pd

# --- 3.2.2 Pelatihan Model Random Forest ---
print("\n--- 3.2.2 Pelatihan Model Random Forest ---")
# Inisialisasi model Random Forest Regressor
# Using random_state for reproducibility
random_forest_model = RandomForestRegressor(random_state=42)

# Latih model menggunakan data pelatihan yang sudah diskalakan
random_forest_model.fit(X_train, y_train)

print("Model Random Forest berhasil dilatih.")

# --- Prediksi pada data uji ---
y_pred_rf = random_forest_model.predict(X_test)

# --- b. Evaluasi kinerja menggunakan MAE, RMSE, MAPE, dan R² ---
print("\n--- Evaluasi Kinerja Model Random Forest pada Data Uji ---")
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

# Define MAPE function (if not already defined in the kernel from previous execution)
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero by replacing 0s in y_true with a small epsilon or filtering
    non_zero_mask = y_true != 0
    if len(y_true[non_zero_mask]) == 0:
        return np.nan # or 0, depending on desired behavior for empty non_zero_mask
    return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100

mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)

print(f"Mean Absolute Error (MAE): {mae_rf:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_rf:.2f}")
print(f"R-squared (R²): {r2_rf:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape_rf:.2f}%")

# --- c. Cross-validation khusus deret waktu untuk memastikan model stabil ---
print("\n--- Time Series Cross-Validation untuk Random Forest ---")
tscv = TimeSeriesSplit(n_splits=5) # Example: 5 splits

# Combine X and y for cross-validation (re-using X_combined, y_combined from LR if available)
# Or redefine if this cell is intended to be run in isolation:
# X_combined = pd.concat([X_train, X_test])
# y_combined = pd.concat([y_train, y_test])

mae_scores_rf = []
rmse_scores_rf = []
r2_scores_rf = []
mape_scores_rf = []

for train_index, val_index in tscv.split(X_combined):
    X_train_cv, X_val_cv = X_combined.iloc[train_index], X_combined.iloc[val_index]
    y_train_cv, y_val_cv = y_combined.iloc[train_index], y_combined.iloc[val_index]

    model_cv = RandomForestRegressor(random_state=42)
    model_cv.fit(X_train_cv, y_train_cv)
    y_pred_cv = model_cv.predict(X_val_cv)

    mae_scores_rf.append(mean_absolute_error(y_val_cv, y_pred_cv))
    rmse_scores_rf.append(np.sqrt(mean_squared_error(y_val_cv, y_pred_cv)))
    r2_scores_rf.append(r2_score(y_val_cv, y_pred_cv))
    mape_scores_rf.append(mean_absolute_percentage_error(y_val_cv, y_pred_cv))

print(f"Average MAE (CV): {np.mean(mae_scores_rf):.2f} (+/- {np.std(mae_scores_rf):.2f})")
print(f"Average RMSE (CV): {np.mean(rmse_scores_rf):.2f} (+/- {np.std(rmse_scores_rf):.2f})")
print(f"Average R-squared (CV): {np.mean(r2_scores_rf):.4f} (+/- {np.std(r2_scores_rf):.4f})")
print(f"Average MAPE (CV): {np.mean(mape_scores_rf):.2f}% (+/- {np.std(mape_scores_rf):.2f}%)")

# --- d. Visualisasi hasil, dengan membandingkan data aktual dan hasil prediksi ---
print("\n--- Visualisasi Prediksi Random Forest vs. Aktual ---")
# Create a DataFrame for easy plotting with original index
results_rf = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_rf}, index=y_test.index)

plt.figure(figsize=(14, 7))
plt.plot(results_rf.index, results_rf['Actual'], label='Harga Aktual', marker='o', linestyle='-', markersize=4)
plt.plot(results_rf.index, results_rf['Predicted'], label='Prediksi Random Forest', marker='x', linestyle='--', markersize=4)
plt.title('Perbandingan Harga Aktual dan Prediksi Random Forest')
plt.xlabel('Tanggal')
plt.ylabel('Harga (IDR)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()